# AE3 - Transformacion y enriquecimiento del dataset de Churn

**Asignatura:** Proyecto Integrado III - Analitica de Datos (PREICA2602B020038)

Segunda etapa del proyecto integrador. Dataset: `scikit-learn/churn-prediction` (Telco Customer Churn).

**Flujo:** dataset de AE2 -> transformacion y validacion -> enriquecimiento -> respuestas a las preguntas de negocio -> tablas de resultados (.csv) y resumen descriptivo (.txt).

Se conserva el mismo proyecto, dataset y repositorio de AE2. El notebook usa rutas relativas y regenera todos los archivos en cada ejecucion.

## 0. Configuracion del entorno (local y Google Colab)

In [1]:
# --- Configuracion de entorno: funciona en local y en Google Colab ---
import sys, pathlib, subprocess

REPO_URL = 'https://github.com/Jeremy-Pedraza/PREICA2602B020038-Analisis-de-Datos.git'

# 'datasets' no viene por defecto en Google Colab; se instala si falta
try:
    import datasets  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)

def _buscar_src():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        for cand in (base / 'src', base / 'EA3' / 'src'):
            if (cand / 'hf_dataset_loader.py').exists():
                return cand.resolve()
    return None

SRC = _buscar_src()
if SRC is None:
    # Entorno tipo Colab: clonar el repo para traer las clases y las carpetas
    repo = pathlib.Path('/content/proyecto_analitica')
    if not repo.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
    SRC = (repo / 'EA3' / 'src').resolve()

EA3_DIR = SRC.parent
RESULTADOS = EA3_DIR / 'resultados'
RESULTADOS.mkdir(parents=True, exist_ok=True)
# Raiz del repositorio: las salidas se muestran relativas a ella
REPO_ROOT = EA3_DIR.parent
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def ruta_rel(p) -> str:
    """Devuelve la ruta relativa a la raiz del repositorio (rutas relativas
    en las salidas al ejecutar el notebook desde el repositorio)."""
    p = pathlib.Path(p).resolve()
    try:
        return p.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return p.as_posix()


from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

print('Clases importadas desde:', ruta_rel(SRC))
print('Resultados en:', ruta_rel(RESULTADOS))

Clases importadas desde: EA3/src
Resultados en: EA3/resultados


## 1. Necesidad, preguntas de negocio y variables (retomadas de AE2)

**Necesidad analitica.** Una compania de telecomunicaciones necesita anticipar el abandono de clientes (*churn*) para orientar acciones de retencion. Cada fila es un cliente; la variable objetivo es `churn`.

En AE2 se planteo la pregunta general: *¿que caracteristicas contractuales, de servicio y de facturacion distinguen a los clientes que cancelan frente a los que permanecen?* En AE3 esa pregunta se descompone en cinco preguntas de negocio operativas, cada una respondida con una tabla:

| # | Pregunta de negocio | Variables necesarias | Resultado esperado |
|---|---------------------|----------------------|--------------------|
| P1 | ¿Que tipo de contrato concentra mayor tasa de cancelacion? | `contract`, `churn` | Tasa de churn por tipo de contrato |
| P2 | ¿Como varia la cancelacion segun la antiguedad del cliente? | `tenure` -> `segmento_antiguedad`, `churn` | Tasa de churn por segmento de antiguedad |
| P3 | ¿Que metodo de pago y servicio de internet se asocian con mayor cancelacion? | `payment_method`, `internet_service`, `churn` | Tasa de churn por metodo de pago y por tipo de internet |
| P4 | ¿Como difieren los cargos entre quienes cancelan y permanecen? | `monthly_charges`, `total_charges`, `cargo_promedio_mensual`, `tenure`, `churn` | Promedios de facturacion por grupo de churn |
| P5 | ¿Influye la cantidad de servicios adicionales contratados en la cancelacion? | add-ons -> `num_servicios_adicionales`, `churn` | Tasa de churn por numero de servicios adicionales |

**Limitacion declarada:** el dataset no tiene marca temporal ni datos de costos, por lo que no permite analisis de tendencia ni de rentabilidad; las respuestas son descriptivas y no causales.

## 2. Cargar el dataset y conservar una version sin modificar

In [2]:
loader = HFDatasetLoader(dataset_id='scikit-learn/churn-prediction', split='train')
df_crudo = loader.to_dataframe()
# Version sin modificar que se conserva para comparar los cambios
df_original = df_crudo.copy()
print('Metadatos:', loader.get_metadata())
df_crudo.head(5)

README.md:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

dataset.csv:   0%|          | 0.00/970k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7043 [00:00<?, ? examples/s]

Metadatos: {'dataset_id': 'scikit-learn/churn-prediction', 'split': 'train', 'config': None, 'revision': None, 'fecha_consulta': '2026-11-10T23:48:08.577964+00:00', 'n_filas': 7043, 'n_columnas': 21, 'es_muestra': False}


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Exportar el crudo tal cual se descargo
ruta_crudo = RESULTADOS / 'churn_crudo.csv'
df_crudo.to_csv(ruta_crudo, index=False, encoding='utf-8')
print('Crudo exportado a:', ruta_rel(ruta_crudo))

# Snapshot ANTES (para la comparacion antes/despues del paso 2)
snapshot_antes = {
    'filas': int(df_crudo.shape[0]),
    'columnas': int(df_crudo.shape[1]),
    'nulos_totales': int(df_crudo.isna().sum().sum()),
    'duplicados': int(df_crudo.duplicated().sum()),
}
print('Snapshot ANTES:', snapshot_antes)

Crudo exportado a: EA3/resultados/churn_crudo.csv
Snapshot ANTES: {'filas': 7043, 'columnas': 21, 'nulos_totales': 0, 'duplicados': 0}


## 3. Transformacion y validacion de los datos

Se aplican las transformaciones de preparacion y se valida cada cambio (que cambio, por que y cuantos registros afecto).

### T1 - Renombrar columnas a snake_case

In [4]:
import re

def a_snake_case(nombre: str) -> str:
    s = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', str(nombre))
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s)
    return s.lower().strip().replace(' ', '_')

df = df_crudo.rename(columns={c: a_snake_case(c) for c in df_crudo.columns})
print('Columnas tras T1:', list(df.columns))

Columnas tras T1: ['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn']


### T2 - Convertir `total_charges` a numerico

`total_charges` viene como texto con cadenas vacias; se convierte con `pd.to_numeric(..., errors='coerce')`.

In [5]:
import pandas as pd
import numpy as np

tipo_antes = str(df['total_charges'].dtype)
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
nulos_tc = int(df['total_charges'].isna().sum())
print(f'total_charges: tipo {tipo_antes} -> {df["total_charges"].dtype}')
print(f'Nulos generados por la conversion: {nulos_tc} '
      f'({nulos_tc / len(df) * 100:.2f}%)')

total_charges: tipo object -> float64
Nulos generados por la conversion: 11 (0.16%)


### T3 - Tratar los nulos de `total_charges`

**Criterio:** los nulos corresponden a clientes con `tenure = 0` (sin facturacion acumulada). Se imputan con `0.0`, valor coherente para un cliente nuevo. No se elimina ningun registro.

In [6]:
tenures_nulos = df.loc[df['total_charges'].isna(), 'tenure'].unique().tolist()
print('tenure de los registros con total_charges nulo:', tenures_nulos)
df['total_charges'] = df['total_charges'].fillna(0.0)
print('Registros imputados:', nulos_tc)
print('Nulos en total_charges tras imputar:', int(df['total_charges'].isna().sum()))

tenure de los registros con total_charges nulo: [0]
Registros imputados: 11
Nulos en total_charges tras imputar: 0


### Validacion: comparacion antes / despues

Control de calidad de filas, columnas, tipos y nulos. No se detectaron duplicados, por lo que no se eliminan registros.

In [7]:
snapshot_despues = {
    'filas': int(df.shape[0]),
    'columnas': int(df.shape[1]),
    'nulos_totales': int(df.isna().sum().sum()),
    'duplicados': int(df.duplicated().sum()),
}
comparacion = pd.DataFrame({'antes': snapshot_antes, 'despues': snapshot_despues})
comparacion['cambio'] = comparacion['despues'] - comparacion['antes']
print(comparacion)
print('\nValidacion duplicados:', 'OK - 0 duplicados' if snapshot_despues['duplicados'] == 0 else 'REVISAR')
print('Validacion nulos:', 'OK - sin nulos' if snapshot_despues['nulos_totales'] == 0 else f"quedan {snapshot_despues['nulos_totales']}")

               antes  despues  cambio
filas           7043     7043       0
columnas          21       21       0
nulos_totales      0        0       0
duplicados         0        0       0

Validacion duplicados: OK - 0 duplicados
Validacion nulos: OK - sin nulos


## 4. Enriquecimiento de la informacion

Se incorporan variables derivadas a partir de las variables existentes (no se usan fuentes externas). Cada una aporta a una pregunta de negocio.

| Variable | Formula / regla | Significado | Utilidad |
|----------|-----------------|-------------|----------|
| `cargo_promedio_mensual` | `total_charges / tenure` (0 si tenure=0) | Cargo medio por mes de antiguedad | P4 |
| `segmento_antiguedad` | `pd.cut(tenure)` en 0-12, 13-24, 25-48, 49+ | Rango de antiguedad | P2 |
| `churn_flag` | `1` si `churn=='Yes'` si no `0` | Objetivo numerico | Agrega la tasa de churn en todas las preguntas |
| `num_servicios_adicionales` | conteo de add-ons == 'Yes' | Cuantos servicios extra contrata | P5 |
| `grupo_cargo_mensual` | `pd.cut(monthly_charges)` en Bajo/Medio/Alto | Nivel de gasto mensual | Perfil de facturacion |

In [8]:
# E1 - cargo_promedio_mensual (derivada de facturacion)
df['cargo_promedio_mensual'] = np.where(
    df['tenure'] > 0,
    df['total_charges'] / df['tenure'].replace(0, np.nan),
    0.0,
)
df['cargo_promedio_mensual'] = df['cargo_promedio_mensual'].fillna(0.0).round(2)

# E2 - segmento_antiguedad (derivada de tenure)
df['segmento_antiguedad'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, float('inf')],
    labels=['0-12', '13-24', '25-48', '49+'],
)

# E3 - churn_flag (objetivo numerico para agregar tasas)
df['churn_flag'] = (df['churn'] == 'Yes').astype(int)

# E4 - num_servicios_adicionales (conteo de add-ons contratados)
ADDONS = ['online_security', 'online_backup', 'device_protection',
          'tech_support', 'streaming_tv', 'streaming_movies']
df['num_servicios_adicionales'] = (df[ADDONS] == 'Yes').sum(axis=1)

# E5 - grupo_cargo_mensual (nivel de gasto mensual)
df['grupo_cargo_mensual'] = pd.cut(
    df['monthly_charges'],
    bins=[-float('inf'), 35, 70, float('inf')],
    labels=['Bajo', 'Medio', 'Alto'],
)

df_transformado = df
print('Columnas nuevas incorporadas:',
      ['cargo_promedio_mensual', 'segmento_antiguedad', 'churn_flag',
       'num_servicios_adicionales', 'grupo_cargo_mensual'])
print('shape final:', df_transformado.shape)
df_transformado[['tenure', 'segmento_antiguedad', 'monthly_charges',
                 'grupo_cargo_mensual', 'num_servicios_adicionales',
                 'churn', 'churn_flag']].head()

Columnas nuevas incorporadas: ['cargo_promedio_mensual', 'segmento_antiguedad', 'churn_flag', 'num_servicios_adicionales', 'grupo_cargo_mensual']
shape final: (7043, 26)


,tenure,segmento_antiguedad,monthly_charges,grupo_cargo_mensual,num_servicios_adicionales,churn,churn_flag
0,1,0-12,29.85,Bajo,1,No,0
1,34,25-48,56.95,Medio,2,No,0
2,2,0-12,53.85,Medio,2,Yes,1
3,45,25-48,42.30,Medio,3,No,0
4,2,0-12,70.70,Alto,0,Yes,1


## 5. Respuestas a las preguntas de negocio

Para cada pregunta se construye una tabla con nombres claros y la tasa de churn (%). La tasa se calcula como promedio de `churn_flag` por grupo.

In [9]:
def tabla_churn_por(col, nombre_col=None):
    nombre_col = nombre_col or col
    g = df_transformado.groupby(col, observed=True)
    t = pd.DataFrame({
        'clientes': g.size(),
        'cancelaciones': g['churn_flag'].sum(),
    })
    t['tasa_churn_%'] = (t['cancelaciones'] / t['clientes'] * 100).round(2)
    t = t.sort_values('tasa_churn_%', ascending=False).reset_index()
    t = t.rename(columns={col: nombre_col})
    return t

tasa_global = df_transformado['churn_flag'].mean() * 100
print(f'Tasa de cancelacion global: {tasa_global:.2f}%  '
      f"({int(df_transformado['churn_flag'].sum())} de {len(df_transformado)} clientes)")

Tasa de cancelacion global: 26.54%  (1869 de 7043 clientes)


### P1 - Churn por tipo de contrato

In [10]:
tabla_p1 = tabla_churn_por('contract')
tabla_p1

,contract,clientes,cancelaciones,tasa_churn_%
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


### P2 - Churn por segmento de antiguedad

In [11]:
tabla_p2 = tabla_churn_por('segmento_antiguedad')
tabla_p2

,segmento_antiguedad,clientes,cancelaciones,tasa_churn_%
0,0-12,2186,1037,47.44
1,13-24,1024,294,28.71
2,25-48,1594,325,20.39
3,49+,2239,213,9.51


### P3 - Churn por metodo de pago y por tipo de internet

In [12]:
tabla_p3a = tabla_churn_por('payment_method')
tabla_p3b = tabla_churn_por('internet_service')
print('Por metodo de pago:'); display(tabla_p3a)
print('Por servicio de internet:'); display(tabla_p3b)

Por metodo de pago:


,payment_method,clientes,cancelaciones,tasa_churn_%
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


Por servicio de internet:


,internet_service,clientes,cancelaciones,tasa_churn_%
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


### P4 - Facturacion comparada entre quienes cancelan y permanecen

In [13]:
g = df_transformado.groupby('churn', observed=True)
tabla_p4 = pd.DataFrame({
    'clientes': g.size(),
    'tenure_prom': g['tenure'].mean().round(2),
    'monthly_charges_prom': g['monthly_charges'].mean().round(2),
    'total_charges_prom': g['total_charges'].mean().round(2),
    'cargo_promedio_mensual_prom': g['cargo_promedio_mensual'].mean().round(2),
}).reset_index()
tabla_p4

,churn,clientes,tenure_prom,monthly_charges_prom,total_charges_prom,cargo_promedio_mensual_prom
0,No,5174,37.57,61.27,2549.91,61.18
1,Yes,1869,17.98,74.44,1531.80,74.43


### P5 - Churn por numero de servicios adicionales

In [14]:
tabla_p5 = tabla_churn_por('num_servicios_adicionales')
tabla_p5

,num_servicios_adicionales,clientes,cancelaciones,tasa_churn_%
0,1,966,442,45.76
1,2,1033,370,35.82
2,3,1118,306,27.37
3,4,852,190,22.30
4,0,2219,475,21.41
5,5,571,71,12.43
6,6,284,15,5.28


### Interpretacion de los hallazgos

- **P1:** el contrato mensual (`Month-to-month`) concentra la mayor tasa de cancelacion; los contratos a uno y dos anios la reducen drasticamente.
- **P2:** la cancelacion es mas alta en clientes nuevos (segmento `0-12` meses) y decae con la antiguedad.
- **P3:** el pago con `Electronic check` y el internet de `Fiber optic` muestran las tasas de churn mas altas.
- **P4:** quienes cancelan tienen menor antiguedad y mayor cargo mensual promedio que quienes permanecen.
- **P5:** una mayor cantidad de servicios adicionales se asocia con menor cancelacion.

**Limitaciones:** sin costos no puede afirmarse rentabilidad; sin marca temporal no hay tendencia; las asociaciones son descriptivas, no causales.

## 6. Exportar el dataset preparado y las tablas de resultados

In [15]:
# Dataset preparado (transformado + enriquecido)
ruta_transformado = RESULTADOS / 'churn_transformado.csv'
df_transformado.to_csv(ruta_transformado, index=False, encoding='utf-8')
print('Transformado exportado a:', ruta_rel(ruta_transformado), '| shape', df_transformado.shape)

# Una tabla .csv por pregunta
tablas = {
    'tabla_p1_churn_por_contrato.csv': tabla_p1,
    'tabla_p2_churn_por_antiguedad.csv': tabla_p2,
    'tabla_p3a_churn_por_metodo_pago.csv': tabla_p3a,
    'tabla_p3b_churn_por_internet.csv': tabla_p3b,
    'tabla_p4_cargos_por_churn.csv': tabla_p4,
    'tabla_p5_churn_por_num_servicios.csv': tabla_p5,
}
for nombre, t in tablas.items():
    ruta = RESULTADOS / nombre
    t.to_csv(ruta, index=False, encoding='utf-8')
    print('Exportada:', ruta_rel(ruta), '->', t.shape)

Transformado exportado a: EA3/resultados/churn_transformado.csv | shape (7043, 26)
Exportada: EA3/resultados/tabla_p1_churn_por_contrato.csv -> (3, 4)
Exportada: EA3/resultados/tabla_p2_churn_por_antiguedad.csv -> (4, 4)
Exportada: EA3/resultados/tabla_p3a_churn_por_metodo_pago.csv -> (4, 4)
Exportada: EA3/resultados/tabla_p3b_churn_por_internet.csv -> (3, 4)
Exportada: EA3/resultados/tabla_p4_cargos_por_churn.csv -> (2, 6)
Exportada: EA3/resultados/tabla_p5_churn_por_num_servicios.csv -> (7, 4)


## 7. Resumen descriptivo automatico (`resumen_dataframe.txt`)

El `.txt` se genera por codigo a partir del DataFrame final (transformado y enriquecido): filas y columnas, nombres y tipos, nulos y estadisticas descriptivas. Se regenera en cada ejecucion.

In [16]:
summarizer = DatasetSummarizer(df_transformado, metadata=loader.get_metadata())
ruta_resumen = summarizer.save(RESULTADOS / 'resumen_dataframe.txt')
print('Resumen guardado en:', ruta_rel(ruta_resumen))

Resumen guardado en: EA3/resultados/resumen_dataframe.txt


In [17]:
# Verificacion: leer e imprimir el .txt generado
print(ruta_resumen.read_text(encoding='utf-8'))

RESUMEN DEL DATASET
1. IDENTIFICACION
------------------------------------------------------------
   dataset_id      : scikit-learn/churn-prediction
   split           : train
   configuracion   : None
   revision        : None
   fecha_consulta  : 2026-11-10T23:48:08.577964+00:00
   es_muestra      : False

2. DIMENSIONES
------------------------------------------------------------
   filas    : 7043
   columnas : 26

3. COLUMNAS Y TIPOS
------------------------------------------------------------
   customer_id                    object
   gender                         object
   senior_citizen                 int64
   partner                        object
   dependents                     object
   tenure                         int64
   phone_service                  object
   multiple_lines                 object
   internet_service               object
   online_security                object
   online_backup                  object
   device_protection              object
   te